<a href="https://colab.research.google.com/github/akemitti/Pred-inad-credito/blob/main/notebook04(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04 — Análise de Sentimento das Atas do COPOM

**Modelos**: BERT Multilingual, FinBERT-PT-BR, TextBlob, NLTK/VADER, Mistral  
**Entrada**: `base_relatorios.csv` (gerado pelo Notebook 03)  
**Saída**: `base_sentimentos.csv` com os scores de todos os modelos

## 0. Instalação de dependências

In [1]:
%pip install -q transformers torch nltk textblob langdetect tqdm
%pip install -q ollama  # necessário apenas se usar Mistral local

## 1. Carregamento da base

In [2]:
import os, re, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# ── Carrega base_relatorios.csv ──────────────────────────────────────────────
# O arquivo pode estar local (gerado pelo NB03) ou no GitHub
URL_BASE = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_relatorios.csv"

try:
    df_docs = pd.read_csv("base_relatorios.csv")
    print("Base carregada do disco local.")
except FileNotFoundError:
    df_docs = pd.read_csv(URL_BASE)
    print("Base carregada do GitHub.")

# Normalizar colunas
df_docs.columns = [c.strip().lower() for c in df_docs.columns]
df_docs["data"] = pd.to_datetime(df_docs["data"], errors="coerce")
for col in ["arquivo", "tipo", "texto"]:
    if col in df_docs.columns:
        df_docs[col] = df_docs[col].fillna("").astype(str)
df_docs["tipo"] = df_docs["tipo"].str.lower().str.strip()

# Filtrar apenas atas do COPOM para análise de sentimento
df_copom = df_docs[df_docs["tipo"] == "copom"].copy().reset_index(drop=True)
print(f"Total de documentos COPOM: {len(df_copom)}")
print(f"Período: {df_copom['data'].min().date()} a {df_copom['data'].max().date()}")
df_copom[["data", "arquivo", "tipo"]].head()


Base carregada do GitHub.
Total de documentos COPOM: 53
Período: 2019-02-06 a 2025-12-10


,data,arquivo,tipo
0,2019-02-06,COPOM220-not20190206220.pdf,copom
1,2019-03-20,COPOM221-not20190320221.pdf,copom
2,2019-05-08,COPOM222-not20190508222.pdf,copom
3,2019-06-19,Copom223-not20190619223.pdf,copom
4,2019-07-31,Copom224-not20190731224.pdf,copom


In [3]:
def preparar_texto(texto: str, max_chars: int = 50_000) -> str:
    """Limpa espaços e trunca para evitar estouro de memória em LLMs."""
    texto = re.sub(r"\s+", " ", str(texto)).strip()
    return texto[:max_chars]


## 2. NLTK / VADER

O VADER é um analisador léxico desenvolvido para inglês. Aqui aplicamos sobre o texto original (em português), o que pode reduzir a acurácia, mas serve como linha de base rápida.

In [4]:
import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

_sid = SentimentIntensityAnalyzer()

def score_vader(texto: str) -> float:
    """Retorna compound [-1, 1]. Positivo = dovish, negativo = hawkish."""
    texto = preparar_texto(texto, 5000)
    if len(texto) < 50:
        return 0.0
    return _sid.polarity_scores(texto)["compound"]

print("Rodando VADER...")
df_copom["score_nltk"] = [score_vader(t) for t in tqdm(df_copom["texto"])]
print(f"VADER concluído. Média: {df_copom['score_nltk'].mean():.3f}")


Rodando VADER...


100%|██████████| 53/53 [00:00<00:00, 73.36it/s]

VADER concluído. Média: -0.809


## 3. TextBlob

TextBlob usa léxico em inglês. Assim como o VADER, serve como baseline rápido.

In [5]:
from textblob import TextBlob

def score_textblob(texto: str) -> float:
    """Retorna polarity [-1, 1]."""
    texto = preparar_texto(texto, 5000)
    if len(texto) < 50:
        return 0.0
    return TextBlob(texto).sentiment.polarity

print("Rodando TextBlob...")
df_copom["score_textblob"] = [score_textblob(t) for t in tqdm(df_copom["texto"])]
print(f"TextBlob concluído. Média: {df_copom['score_textblob'].mean():.3f}")


Rodando TextBlob...


100%|██████████| 53/53 [00:01<00:00, 43.76it/s]

TextBlob concluído. Média: 0.012


## 4. BERT Multilingual

**Modelo**: `nlptown/bert-base-multilingual-uncased-sentiment`  
Classifica em 1–5 estrelas; convertemos para [-1, 1]: `score = (estrelas - 3) / 2`.

In [6]:
import torch
from transformers import pipeline as hf_pipeline

_bert_model = "nlptown/bert-base-multilingual-uncased-sentiment"
_device_id   = 0 if torch.cuda.is_available() else -1

_bert_pipe = hf_pipeline(
    "sentiment-analysis",
    model=_bert_model,
    tokenizer=_bert_model,
    device=_device_id,
    truncation=True,
    max_length=512,
)
print(f"BERT carregado  (device={'GPU' if torch.cuda.is_available() else 'CPU'})")

def _bert_chunk_score(texto: str, chunk_words: int = 350, max_chunks: int = 8) -> float:
    """Divide o texto em chunks de ~350 palavras e calcula a média dos scores."""
    palavras = texto.split()
    chunks = [" ".join(palavras[i:i+chunk_words])
              for i in range(0, len(palavras), chunk_words)][:max_chunks]
    scores = []
    for chunk in chunks:
        if len(chunk) < 50:
            continue
        try:
            res   = _bert_pipe(chunk[:512])[0]
            stars = int(res["label"].split()[0])   # '3 stars' -> 3
            scores.append((stars - 3) / 2)
        except Exception:
            pass
    return float(np.mean(scores)) if scores else 0.0

def score_bert(texto: str) -> float:
    texto = preparar_texto(texto, 50_000)
    if len(texto) < 50:
        return 0.0
    return _bert_chunk_score(texto)

print("Rodando BERT Multilingual...")
df_copom["score_bert"] = [score_bert(t) for t in tqdm(df_copom["texto"])]
print(f"BERT concluído. Média: {df_copom['score_bert'].mean():.3f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT carregado  (device=CPU)
Rodando BERT Multilingual...


100%|██████████| 53/53 [04:20<00:00,  4.91s/it]

BERT concluído. Média: -0.170


## 5. FinBERT-PT-BR

**Modelo**: `lucas-leme/FinBERT-PT-BR`  
BERT pré-treinado em textos financeiros em português. Mapa de labels: positivo → +1, neutro → 0, negativo → −1.

In [7]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForSequenceClassification

_finbert_name = "lucas-leme/FinBERT-PT-BR"
_device       = "cuda" if torch.cuda.is_available() else "cpu"

_finbert_tok   = AutoTokenizer.from_pretrained(_finbert_name)
_finbert_model = BertForSequenceClassification.from_pretrained(_finbert_name).to(_device).eval()

_MAX_LEN_FINBERT   = 512
_CHUNK_BODY_FINBERT = _MAX_LEN_FINBERT - 2
_MAX_CHUNKS_FINBERT = 20

def _pt_label_finbert(i: int) -> str:
    l = str(_finbert_model.config.id2label.get(i, "")).lower()
    if "pos" in l: return "positivo"
    if "neg" in l: return "negativo"
    return "neutro"

def score_finbert(texto: str) -> float:
    """Score [-1, 1]: +1 = positivo/dovish, -1 = negativo/hawkish."""
    texto = preparar_texto(texto, 50_000).strip()
    if not texto or len(texto) < 50:
        return 0.0
    ids = _finbert_tok.encode(texto, add_special_tokens=False)
    if not ids:
        return 0.0
    chunks = [ids[i:i+_CHUNK_BODY_FINBERT]
              for i in range(0, len(ids), _CHUNK_BODY_FINBERT)][:_MAX_CHUNKS_FINBERT]
    probs_sum = None
    for ch in chunks:
        chunk_text = _finbert_tok.decode(ch, clean_up_tokenization_spaces=True)
        enc = _finbert_tok(chunk_text, return_tensors="pt",
                           truncation=True, max_length=_MAX_LEN_FINBERT)
        enc = {k: v.to(_device) for k, v in enc.items()}
        with torch.no_grad():
            logits = _finbert_model(**enc).logits
            p = F.softmax(logits, dim=-1).squeeze(0).cpu()
        probs_sum = p if probs_sum is None else (probs_sum + p)

    probs  = probs_sum / len(chunks)
    scores = {_pt_label_finbert(i): float(probs[i]) for i in range(probs.shape[-1])}
    return scores.get("positivo", 0.0) - scores.get("negativo", 0.0)

print("Rodando FinBERT-PT-BR...")

resultados = []

try:
    for texto in tqdm(df_copom["texto"]):
        resultados.append(score_finbert(texto))

except KeyboardInterrupt:
    print("\nExecução interrompida manualmente. Salvando progresso parcial...")

df_copom.loc[:len(resultados)-1, "score_finbert"] = resultados

print(f"Textos processados: {len(resultados)} de {len(df_copom)}")

if len(resultados) > 0:
    print(f"Média parcial: {df_copom['score_finbert'].dropna().mean():.3f}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Rodando FinBERT-PT-BR...


  0%|          | 0/53 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

100%|██████████| 53/53 [13:07<00:00, 14.86s/it]

Textos processados: 53 de 53
Média parcial: -0.404


## 7. Salvar base com todos os scores (ex Mistral)

In [8]:
# Colunas de score disponíveis
score_cols = ["score_nltk", "score_textblob", "score_bert", "score_finbert"]

# Remover modelos com todos NaN (ex.: Mistral sem Ollama)
score_cols_validos = [c for c in score_cols if df_copom[c].notna().any()]
print("Modelos com scores válidos:", score_cols_validos)

# Salvar
df_copom.to_csv("/content/sample_data/base_sentimentos_copom_sem_mistral.csv", index=False)
print("base_sentimentos.csv salvo com sucesso.")
df_copom[["data", "arquivo"] + score_cols_validos].head()


Modelos com scores válidos: ['score_nltk', 'score_textblob', 'score_bert', 'score_finbert']
base_sentimentos.csv salvo com sucesso.


,data,arquivo,score_nltk,score_textblob,score_bert,score_finbert
0,2019-02-06,COPOM220-not20190206220.pdf,-0.7783,0.0000,-0.071429,-0.248734
1,2019-03-20,COPOM221-not20190320221.pdf,-0.6808,0.0000,-0.062500,-0.336417
2,2019-05-08,COPOM222-not20190508222.pdf,-0.9274,0.0625,-0.125000,-0.388865
3,2019-06-19,Copom223-not20190619223.pdf,-0.7783,0.0000,-0.187500,-0.407422
4,2019-07-31,Copom224-not20190731224.pdf,-0.8402,0.0000,0.062500,-0.227865
